<a href="https://colab.research.google.com/github/gifeonu-PGee/enog-server/blob/main/PROJECT_02_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PROJECT 02 — CATS VS DOGS IMAGE CLASSIFICATION

Objective: Build a convolutional neural network that can examine an image and classify it as either a cat or a dog.

Problem type: Binary image classification

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os
import zipfile

In [ ]:
!pip install -U protobuf tensorflow-datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

# Load cats_vs_dogs, splitting 80% train / 20% validation
(train_ds, val_ds), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True,
)

print("Classes:", metadata.features['label'].names)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.SSAPF6_4.0.1/cats_vs_dogs-train.tfrecord-[0-9][0-9…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Classes: ['cat', 'dog']


tfds.load(...) — pulls the cats_vs_dogs dataset, split 80% train / 20% validation

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, label

train_ds = train_ds.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

preprocess() — resizes every image to 150×150 and scales pixel values from 0–255 down to 0–1 (neural nets train better on small numbers).

.shuffle().batch().prefetch() — shuffles training data so the model doesn't learn order patterns, groups images into batches of 32, and prefetches the next batch while the current one trains (speed optimization).

In [ ]:
model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # binary output
])

model.summary()

The CNN architecture

Input(shape=(150,150,3)) — the entry point: 150×150 pixels, 3 color channels (RGB).

Conv2D(32, (3,3), activation='relu') — slides 32 filters over the image to detect simple features (edges, colors, textures).

ReLU adds non-linearity so it can learn complex patterns, not just straight lines.

MaxPooling2D(2,2) — shrinks the image by keeping only the strongest signal in each 2×2 patch. Reduces size, keeps the important stuff.

This Conv → Pool pair repeats 4 times, with filter counts increasing (32 → 64 → 128 → 128). Early layers catch simple shapes; deeper layers combine them into complex features (fur texture, ear shapes, etc.).

Flatten() — squashes the final 3D feature maps into one long 1D list of numbers, so it can feed into regular dense layers.

Dropout(0.5) — randomly "turns off" half the neurons during training. Prevents the model from memorizing the training images instead of learning general patterns.

Dense(512, activation='relu') — a fully-connected layer that combines all the extracted features to start making a decision.

Dense(1, activation='sigmoid') — the output layer. One key difference from your CIFAR-10 project: CIFAR-10 had 10 classes, so it used softmax to output probabilities across 10 categories. Here we only have 2 classes (cat/dog), so we use sigmoid instead — it outputs a single number between 0 and 1 (e.g., 0.85 = 85% confident it's a dog).

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True
)

Training

binary_crossentropy — the loss function paired with sigmoid, used specifically for two-class problems (CIFAR-10 would've used categorical_crossentropy or sparse_categorical_crossentropy).

In [ ]:
history = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 663s 1s/step - accuracy: 0.7779 - loss: 0.4713 - val_accuracy: 0.8029 - val_loss: 0.4270
Epoch 2/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 692s 1s/step - accuracy: 0.8055 - loss: 0.4233 - val_accuracy: 0.8106 - val_loss: 0.4238
Epoch 3/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 647s 1s/step - accuracy: 0.8288 - loss: 0.3828 - val_accuracy: 0.8399 - val_loss: 0.3660
Epoch 4/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 665s 1s/step - accuracy: 0.8436 - loss: 0.3552 - val_accuracy: 0.8461 - val_loss: 0.3461
Epoch 5/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 687s 1s/step - accuracy: 0.8581 - loss: 0.3311 - val_accuracy: 0.8571 - val_loss: 0.3320
Epoch 6/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 644s 1s/step - accuracy: 0.8700 - loss: 0.3031 - val_accuracy: 0.8626 - val_loss: 0.3179
Epoch 7/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 644s 1s/step - accuracy: 0.8779 - loss: 0.2848 - val_accuracy: 0.8678 - val_loss: 0.3115
Epoch 8/20
582/582 ━━━━━━━━━━━━━━━━━━━━ 677s 1s/step - accuracy: 0.8903 - loss: 0.2662 - val_accu

model.fit(train_ds, epochs=20, validation_data=val_ds) — trains for 20 full passes through the data, checking performance on unseen validation images after each pass.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend()
plt.title('Loss')

plt.show()

In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np

uploaded = files.upload()

for fn in uploaded.keys():
    img = image.load_img(fn, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0][0]
    label = "Dog 🐶" if prediction > 0.5 else "Cat 🐱"
    confidence = prediction if prediction > 0.5 else 1 - prediction

    print(f"{fn}: {label} (confidence: {confidence:.2%})")

In [ ]:
model.save("cats_vs_dogs_model.keras")

print("Model saved successfully!")

This is to Save the trained model

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("cats_vs_dogs_model.keras")

print("Model loaded successfully!")

In [ ]:
prediction = model.predict(img_array)[0][0]

if prediction > 0.5:
    label = "Dog"
    confidence = prediction
else:
    label = "Cat"
    confidence = 1 - prediction

print("="*40)
print(f"Prediction : {label}")
print(f"Confidence : {confidence:.2%}")
print("="*40)

In [ ]:
plt.imshow(img)
plt.axis("off")
plt.title("Uploaded Image")
plt.show()

In [ ]:
print()

if confidence > 0.95:
    print("The model is highly confident.")
elif confidence > 0.80:
    print("The model is reasonably confident.")
else:
    print("The model is uncertain. Consider another image.")

In [ ]:
import pandas as pd
from datetime import datetime

In [ ]:
result = pd.DataFrame({
    "Time":[datetime.now()],
    "Prediction":[label],
    "Confidence":[confidence]
})

result.to_csv("prediction_log.csv",
              mode="a",
              header=False,
              index=False)

print(result)